# Forest-fire regression

An undergraduate linear-regression exercise using the UCI Forest Fires data. The target is burned area (`area`, hectares), selected by name before categorical encoding. The original assignment selected the final column after `get_dummies`, which could accidentally choose an encoded day/category instead of the target.

This version uses an 80/20 random split (`random_state=1`), fits month/day encoding on training rows only, and compares test MSE with a training-mean baseline. It models raw area, not the paper's log-transformed target. Random row splits, a small dataset and skewed outcomes limit what this experiment establishes. It is not an operational wildfire prediction service.


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error


In [ ]:
repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "notebooks/forest_fire_regression.ipynb").is_file()), None
)
if repo_root is None:
    raise FileNotFoundError("Open the notebook from this project or its notebooks directory.")
data_root = Path(os.environ.get("ML_DATA_DIR") or repo_root / "data/raw").expanduser()
if not data_root.is_absolute():
    data_root = repo_root / data_root
data_path = data_root / "forest-fires/forestfires.csv"
if not data_path.is_file():
    raise FileNotFoundError("Run python scripts/download_forest_fires.py first.")
dataset = pd.read_csv(data_path)
X = dataset.drop(columns="area")
y = dataset["area"]
print(f"Records: {len(dataset)}; target: {y.name}; input columns: {X.shape[1]}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=1
)
preprocess = ColumnTransformer(
    [("calendar", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["month", "day"])],
    remainder="passthrough",
)
model = make_pipeline(preprocess, LinearRegression())
model.fit(X_train, y_train)
print(f"Train rows: {len(X_train)}; test rows: {len(X_test)}")


In [ ]:
y_pred = model.predict(X_test)
baseline_pred = np.full(len(y_test), y_train.mean())
mse = mean_squared_error(y_test, y_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
print(f"Test MSE (squared hectares): {mse:.2f}")
print(f"Training-mean baseline MSE: {baseline_mse:.2f}")
print(f"Negative linear predictions: {int((y_pred < 0).sum())}")


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, label="Linear model")
upper = max(float(y_test.max()), float(y_pred.max()))
plt.plot([0, upper], [0, upper], color="black", linestyle="--", label="Ideal prediction")
plt.xlabel("Actual burned area (hectares)")
plt.ylabel("Predicted burned area (hectares)")
plt.title("Forest fires: held-out predictions")
plt.legend()
plt.tight_layout()
plt.show()


## Interpretation

MSE is sensitive to large fires. A poor score or negative prediction is useful evidence of the limitations of this ordinary linear model, not a value to hide by trimming test rows. The training-mean comparison uses no held-out targets. The categorical encoder is fitted inside the training pipeline. Compare results only for the stated input/split and do not treat them as an independently validated forecast.

Dataset citation and acquisition instructions are in `data/README.md`. This notebook adapts Carl James Poopalaretnam's original PS2 college exercise; saved original outputs are not carried into the curated version.
